In [0]:
from pyspark.sql import functions as F

# STEP 1: READ DATA
df = spark.table("novacart_dev.bronze.customers")

In [0]:
# STEP 2: STANDARDIZE COLUMN NAMES
df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

# STEP 3: GLOBAL NULL HANDLING
df = df.replace(["\\N", "?", "", "null", "NULL"], None)

# STEP 4: TRIM + LOWER STRING COLUMNS
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.lower(F.col(col))))

# STEP 5: DATE PARSING (MULTIPLE FORMATS)

df = df.withColumn(
    "registration_date",
    F.coalesce(
        F.to_date(F.expr("try_to_timestamp(registration_date, 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(registration_date, 'yyyy/MM/dd')")),
        F.to_date(F.expr("try_to_timestamp(registration_date, 'dd/MM/yyyy')"))
    )
)

# STEP 6: REMOVE DUPLICATES
df = df.dropDuplicates(["customer_id"])

# STEP 7: VALIDATION (CHANNEL)
df = df.withColumn(
    "channel",
    F.when(F.col("channel").isin("web", "mobile"), F.col("channel"))
     .otherwise(None)
)



In [0]:
# STEP 8: DATA QUALITY FLAGS
df = df.withColumn(
    "dq_missing_customer_id",
    F.when(F.col("customer_id").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_missing_email",
    F.when(F.col("email").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_channel",
    F.when(~F.col("channel").isin("web", "mobile"), 1).otherwise(0)
)

# STEP 9: ADD LOAD TIMESTAMP
df = df.withColumn("load_timestamp", F.current_timestamp())

In [0]:
# STEP 10: WRITE TO SILVER (S3 PATH OPTIONAL BUT RECOMMENDED)
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.silver.slv_customers")

display("novacart_dev.silver.slv_customers")